# 12 · Skill Gap Engine (Severity Classification & Gap Inventory)

**Project:** Enterprise HR AI  

> ### ⚠️ PROMINENT DATA INTEGRITY WARNING
> **SYNTHETIC DATA — employee current-skill possession was not present in any source file and has been simulated using a tenure/training-based heuristic for MVP demonstration purposes only. This must NOT be presented to stakeholders as real observed skill data. Real deployment requires an actual skills inventory (HRIS export, LMS completion records, or self-assessment survey).**

---

---
## Step 1 · Load Synthetic Skills & Perform 30-Employee Fidelity Sanity Check

Before utilizing the synthetic skill inventory, we evaluate statistical fidelity by sampling 30 random employees (`random_state=42`), computing their realized skill possession rate (possessed skills / 10 required skills), and comparing it against their theoretical possession probability from Step 15's formula:
$$\text{possession\_probability} = \min(0.30 + 0.05 \times \text{YearsAtCompany} + 0.05 \times \text{TrainingTimesLastYear}, 0.95)$$

If Pearson correlation $r < 0.50$, execution must halt with an explicit data quality warning.

In [1]:
import pandas as pd
import numpy as np
import os

PROC = os.path.join('..', 'data', 'processed')
skills_file = os.path.join(PROC, 'employee_skills_synthetic.csv')
att_file = os.path.join(PROC, 'employee_attrition_processed.csv')
profiles_file = os.path.join(PROC, 'role_skill_profiles.csv')

# Load synthetic skills ignoring comment header
df_skills = pd.read_csv(skills_file, comment='#')
df_att = pd.read_csv(att_file)
df_profiles = pd.read_csv(profiles_file)

print(f'Loaded synthetic skills records : {len(df_skills):,}')
print(f'Loaded employee anchor records   : {len(df_att):,}')
print(f'Loaded role benchmark profiles   : {len(df_profiles):,}')

# Identify and isolate Manager employees (102 records)
mgr_ids = set(df_att[df_att['JobRole'] == 'Manager']['EmployeeNumber'])
non_mgr_att = df_att[~df_att['EmployeeNumber'].isin(mgr_ids)].copy()
non_mgr_skills = df_skills[~df_skills['EmployeeNumber'].isin(mgr_ids)].copy()

print(f'\nNon-Manager employees to evaluate: {len(non_mgr_att):,} (102 Managers excluded)')
print(f'Non-Manager skill records        : {len(non_mgr_skills):,}')

# Sample 30 non-manager employees
sample_ids = non_mgr_att['EmployeeNumber'].sample(n=30, random_state=42).tolist()
sample_att = non_mgr_att[non_mgr_att['EmployeeNumber'].isin(sample_ids)].copy()

# Recompute theoretical probability
sample_att['theoretical_prob'] = sample_att.apply(
    lambda r: min(0.30 + 0.05 * r['YearsAtCompany'] + 0.05 * r['TrainingTimesLastYear'], 0.95),
    axis=1
)

# Compute realized possession rate
sample_realized = (
    non_mgr_skills[non_mgr_skills['EmployeeNumber'].isin(sample_ids)]
    .groupby('EmployeeNumber')
    .agg(total_req=('has_skill', 'count'), possessed=('has_skill', 'sum'))
    .reset_index()
)
sample_realized['realized_rate'] = sample_realized['possessed'] / sample_realized['total_req']

check_df = sample_att.merge(sample_realized, on='EmployeeNumber')
corr_val = check_df['theoretical_prob'].corr(check_df['realized_rate'])

print('=' * 80)
print(f'30-EMPLOYEE SANITY CHECK CORRELATION (Theoretical vs Realized): r = {corr_val:.4f}')
print('=' * 80)

if corr_val < 0.50:
    raise ValueError(f'DATA QUALITY CONCERN: Realized rate correlation ({corr_val:.4f}) is below 0.50 threshold!')
else:
    print('CONFIRMED: Strong statistical fidelity (r >= 0.50). Synthetic data generation is sound.')

print('\nFirst 5 Sample Records:')
for _, r in check_df.head(5).iterrows():
    print(f'  Emp #{r["EmployeeNumber"]} ({r["JobRole"]}): Tenure={r["YearsAtCompany"]}y, Training={r["TrainingTimesLastYear"]}x | Theory={r["theoretical_prob"]:.2f}, Realized={r["realized_rate"]:.2f} ({r["possessed"]}/{r["total_req"]})')

Loaded synthetic skills records : 13,782
Loaded employee anchor records   : 1,470
Loaded role benchmark profiles   : 9

Non-Manager employees to evaluate: 1,368 (102 Managers excluded)
Non-Manager skill records        : 13,680
30-EMPLOYEE SANITY CHECK CORRELATION (Theoretical vs Realized): r = 0.8501
CONFIRMED: Strong statistical fidelity (r >= 0.50). Synthetic data generation is sound.

First 5 Sample Records:
  Emp #60 (Sales Executive): Tenure=9y, Training=2x | Theory=0.85, Realized=1.00 (10/10)
  Emp #68 (Sales Executive): Tenure=4y, Training=2x | Theory=0.60, Realized=0.70 (7/10)
  Emp #94 (Research Scientist): Tenure=5y, Training=3x | Theory=0.70, Realized=0.60 (6/10)
  Emp #104 (Laboratory Technician): Tenure=10y, Training=1x | Theory=0.85, Realized=0.90 (9/10)
  Emp #162 (Research Director): Tenure=20y, Training=5x | Theory=0.95, Realized=1.00 (10/10)


---
## Step 2 · Compute Individual Skill Gaps & Severity Classification

For each of the 1,368 non-manager employees:
1. **Missing Skills:** List all benchmark skills where `has_skill == 0`.
2. **Gap Metrics:** Compute `gap_count` (missing count) and `gap_percentage` ($gap\_count / total\_required$).
3. **Severity Thresholds:**
   - **HIGH Severity:** $\text{gap\_percentage} \ge 70\%$ (missing 7+ of 10 skills)
   - **MEDIUM Severity:** $40\% \le \text{gap\_percentage} < 70\%$ (missing 4 to 6 skills)
   - **LOW Severity:** $\text{gap\_percentage} < 40\%$ (missing 0 to 3 skills)

In [2]:
gap_records = []
emp_role_map = dict(zip(df_att['EmployeeNumber'], df_att['JobRole']))

for emp_id, group in non_mgr_skills.groupby('EmployeeNumber'):
    role = emp_role_map[emp_id]
    total_req = len(group)
    missing_list = group[group['has_skill'] == 0]['skill_name'].tolist()
    gap_cnt = len(missing_list)
    gap_pct = gap_cnt / total_req if total_req > 0 else 0.0
    
    # Explicit severity threshold classification
    if gap_pct >= 0.70:
        severity = 'HIGH'
    elif gap_pct >= 0.40:
        severity = 'MEDIUM'
    else:
        severity = 'LOW'
        
    gap_records.append({
        'EmployeeNumber': emp_id,
        'JobRole': role,
        'missing_skills': '; '.join(missing_list) if missing_list else 'None',
        'gap_count': gap_cnt,
        'total_required': total_req,
        'gap_percentage': round(gap_pct, 4),
        'severity': severity
    })

df_gaps = pd.DataFrame(gap_records)
print(f'Total skill gap profiles generated: {len(df_gaps):,} employees')
assert len(df_gaps) == 1368, f'Expected 1,368 non-manager gap profiles, got {len(df_gaps)}'

# Severity distribution
sev_counts = df_gaps['severity'].value_counts()
sev_pcts = (df_gaps['severity'].value_counts(normalize=True) * 100).round(2)

sev_summary = pd.DataFrame({
    'Headcount': sev_counts,
    'Percentage (%)': sev_pcts
})
sev_summary.index.name = 'Severity Band'

print('\n=== SKILL GAP SEVERITY DISTRIBUTION (n=1,368) ===')
print(sev_summary.to_string())

print('\nSample Profiles Across Severity Bands:')
for band in ['HIGH', 'MEDIUM', 'LOW']:
    ex_row = df_gaps[df_gaps['severity'] == band].iloc[0]
    print(f'\n[{band} SEVERITY EXAMPLE] Employee #{ex_row["EmployeeNumber"]} ({ex_row["JobRole"]})')
    print(f'  Missing Skills Count : {ex_row["gap_count"]} / {ex_row["total_required"]} ({ex_row["gap_percentage"]*100:.1f}%)')
    print(f'  Missing Skills List  : {ex_row["missing_skills"]}')

Total skill gap profiles generated: 1,368 employees

=== SKILL GAP SEVERITY DISTRIBUTION (n=1,368) ===
               Headcount  Percentage (%)
Severity Band                           
LOW                  870           63.60
MEDIUM               427           31.21
HIGH                  71            5.19

Sample Profiles Across Severity Bands:

[HIGH SEVERITY EXAMPLE] Employee #39 (Laboratory Technician)
  Missing Skills Count : 10 / 10 (100.0%)
  Missing Skills List  : Reading Comprehension; Active Listening; Critical Thinking; Speaking; Science; Google Docs; MEDITECH software; Microsoft Excel; Microsoft Office software; Microsoft Outlook

[MEDIUM SEVERITY EXAMPLE] Employee #1 (Sales Executive)
  Missing Skills Count : 5 / 10 (50.0%)
  Missing Skills List  : Speaking; Reading Comprehension; Bentley MicroStation; Eclipse IDE; Facebook

[LOW SEVERITY EXAMPLE] Employee #2 (Research Scientist)
  Missing Skills Count : 1 / 10 (10.0%)
  Missing Skills List  : Reading Comprehension


---
## Step 3 · Save Skill Gap Inventory with Warning Comment Header

Target file: `data/processed/employee_skill_gaps.csv`  
The synthetic data caveat is explicitly preserved in line 1 of the export.

In [3]:
out_path = os.path.join(PROC, 'employee_skill_gaps.csv')

warning_comment = (
    '# SYNTHETIC DATA — employee current-skill possession was not present in any source file '
    'and has been simulated using a tenure/training-based heuristic for MVP demonstration purposes only. '
    'This must NOT be presented to stakeholders as real observed skill data. Real deployment requires '
    'an actual skills inventory (HRIS export, LMS completion records, or self-assessment survey).\n'
)

with open(out_path, 'w', encoding='utf-8') as f:
    f.write(warning_comment)
    df_gaps.to_csv(f, index=False)

file_size = os.path.getsize(out_path)
print(f'Saved skill gaps file to : {out_path}')
print(f'File size                : {file_size:,} bytes')
print(f'Total rows               : {len(df_gaps):,} records')

# Round-trip reload validation
df_check = pd.read_csv(out_path, comment='#')
assert len(df_check) == 1368, 'Row count mismatch on reload!'
assert list(df_check.columns) == ['EmployeeNumber', 'JobRole', 'missing_skills', 'gap_count', 'total_required', 'gap_percentage', 'severity']
print('CONFIRMED: Round-trip read verified cleanly with comment handling.')

Saved skill gaps file to : ..\data\processed\employee_skill_gaps.csv
File size                : 126,692 bytes
Total rows               : 1,368 records
CONFIRMED: Round-trip read verified cleanly with comment handling.
